<a href="https://colab.research.google.com/github/fajareka1/PCVK_Ganjil_2026_v2/blob/main/Modul_5_Praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Fajar Eka Sandiyuda'
NIM  = '244107020043'

N     = int(NIM[-3:])
PARAM = {
    'bins' : 2 ** ((N % 4) + 3),
    'clip' : round(1.0 + (N % 5) * 0.5, 1),
    'tile' : (N % 4) + 2,
    'L'    : 2 ** ((N % 3) + 1),
}
waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))

print('NAMA  :', NAMA)
print('NIM   :', NIM, '| N =', N)
for k, v in PARAM.items():
    print('%-6s :' % k, v)
print('WAKTU :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM:', sys.version.split()[0], platform.platform())
kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN :', hashlib.sha256(kunci.encode()).hexdigest()[:16])

NAMA  : Fajar Eka Sandiyuda
NIM   : 244107020043 | N = 43
bins   : 64
clip   : 2.5
tile   : 5
L      : 4
WAKTU : 2026-09-29 18:50:06 WIB
SISTEM: 3.13.15 Linux-6.6.122+-x86_64-with-glibc2.39
TOKEN : d699d0bcf8fe4237


# Week 5 – Histogram, Histogram Equalization, dan Dithering

Notebook ini mengerjakan Modul 5 PCVK. Semua percobaan dijalankan dua tahap: **tahap 1** memakai citra contoh (`lena.jpg`, `lena_lc.jpg`) untuk memastikan kode benar, lalu **tahap 2** memakai citra KTM sendiri (`KTM1a` sampai `KTM1d`) yang dianalisis.

Parameter pribadi dihitung dari tiga digit terakhir NIM (N = 43), dan hasilnya sudah tercetak di sel identitas di atas: `bins = 64`, `clip = 2.5`, `tile = 5`, `L = 4`.

**Sebelum menjalankan:** unggah `lena.jpg` dan `lena_lc.jpg` ke `MyDrive/PCVK/Images/`, lalu `KTM1a.jpg` sampai `KTM1d.jpg` ke `MyDrive/PCVK/Images/244107020043/`. KTM sebaiknya sudah dicrop tanpa background.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import math, os, glob, time
import pandas as pd
from IPython.display import display
from skimage import io
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    pass

# hubungkan Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Bukan di Colab, langkah mount Drive dilewati.')

CONTOH = '/content/drive/MyDrive/PCVK/Images'            # citra contoh bersama
BASE   = '/content/drive/MyDrive/PCVK/Images/' + NIM     # citra KTM milik Anda

# ---- pengaturan tambahan ----
MAKS_SISI   = 1000     # foto HP sangat besar; sisi terpanjang diperkecil ke sini supaya perulangan manual tidak lama
SENSOR_AKTIF = False   # ubah ke True sebelum push ke repo PUBLIK: area foto pada KTM akan ditutup pada tampilan
SENSOR_BOX   = [(0.03, 0.22, 0.30, 0.78)]   # (x1, y1, x2, y2) sebagai pecahan 0-1 dari lebar/tinggi kartu -> sesuaikan letak foto di KTM Anda
ROI_TEKS     = (0.40, 0.90, 0.32, 0.98)     # (y1, y2, x1, x2) area tulisan NIM/nama pada KTM -> sesuaikan
ROI_LATAR    = (0.05, 0.22, 0.35, 0.95)     # (y1, y2, x1, x2) area latar polos (tanpa tulisan) untuk mengukur noise -> sesuaikan

In [ ]:
def muat(path, gray=False):
    """Baca citra; kalau terlalu besar, kecilkan (INTER_AREA) agar perulangan manual tidak lambat."""
    if not os.path.exists(path):
        raise FileNotFoundError('Berkas tidak ditemukan: ' + path +
                                '\nPastikan sudah diunggah ke Drive dan path-nya benar.')
    img = cv.imread(path, cv.IMREAD_GRAYSCALE if gray else cv.IMREAD_COLOR)
    if img is None:
        raise ValueError('Gagal membaca citra: ' + path)
    h, w = img.shape[:2]
    s = max(h, w)
    if s > MAKS_SISI:
        k = MAKS_SISI / s
        img = cv.resize(img, (round(w * k), round(h * k)), interpolation=cv.INTER_AREA)
    return img

def tutup_area(img):
    out = img.copy()
    H, W = out.shape[:2]
    for (x1, y1, x2, y2) in SENSOR_BOX:
        out[int(y1 * H):int(y2 * H), int(x1 * W):int(x2 * W)] = 0
    return out

def tampil(ax, img, judul, ktm=False):
    """Tampilkan citra (gray/BGR) dengan judul yang memuat NIM."""
    im = tutup_area(img) if (ktm and SENSOR_AKTIF) else img
    if im.ndim == 2:
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    else:
        ax.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
    ax.set_title(NIM + ' - ' + judul, fontsize=9)
    ax.axis('off')

def potong(img, roi):
    H, W = img.shape[:2]
    y1, y2, x1, x2 = roi
    return img[int(y1 * H):int(y2 * H), int(x1 * W):int(x2 * W)]

def p_cdf(gray):
    h = np.bincount(gray.ravel(), minlength=256)
    p = h / gray.size
    return p, np.cumsum(p)

BERKAS_KTM = ['KTM1a', 'KTM1b', 'KTM1c', 'KTM1d']
print('Siap. Folder contoh :', CONTOH)
print('      Folder KTM    :', BASE)

### Cek area ROI pada KTM
Kotak **hijau** = area tulisan (NIM/nama), kotak **merah** = area latar polos. Keduanya dipakai untuk menilai keterbacaan dan noise di beberapa percobaan nanti. Kalau posisinya kurang pas pada KTM Anda, ubah `ROI_TEKS` / `ROI_LATAR` di sel pengaturan, lalu jalankan ulang sel itu dan sel ini.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))
for ax, n in zip(axs, BERKAS_KTM):
    p = BASE + '/' + n + '.jpg'
    if not os.path.exists(p):
        ax.set_title('tidak ditemukan: ' + n); ax.axis('off'); continue
    im = muat(p)
    tampil(ax, im, n, ktm=True)
    H, W = im.shape[:2]
    for roi, warna in [(ROI_TEKS, 'lime'), (ROI_LATAR, 'red')]:
        y1, y2, x1, x2 = roi
        ax.add_patch(plt.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H,
                                   fill=False, ec=warna, lw=2))
plt.show()

---
# E-1 Percobaan Histogram

In [ ]:
def histogram_manual(img, L=256):
    h = np.zeros(L, dtype=np.int64)          # 1. siapkan penghitung, isi 0
    for y in range(img.shape[0]):            # 2. telusuri setiap baris
        for x in range(img.shape[1]):        #    dan setiap kolom
            h[img[y, x]] += 1                # 3. tambah 1 sesuai nilai piksel
    return h

def plot_hist_rgb(bgr, judul):
    fig, axs = plt.subplots(1, 3, figsize=(20, 5), sharex=True, sharey=True)
    fig.suptitle(NIM + ' - Histogram RGB ' + judul)
    for ax, (idx, nama, warna) in zip(axs, [(2, 'Merah', 'red'), (1, 'Hijau', 'green'), (0, 'Biru', 'blue')]):
        h = histogram_manual(bgr[:, :, idx])
        assert h.sum() == bgr.shape[0] * bgr.shape[1]      # pemeriksaan wajib
        ax.bar(np.arange(256), h, color=warna)
        ax.set_title(nama)
    fig.text(0.09, 0.5, 'Jumlah Kemunculan', va='center', rotation='vertical')
    fig.text(0.5, 0.04, 'Intensitas Warna', ha='center')
    plt.show()

def plot_hist_gray(gray, judul):
    h = histogram_manual(gray)
    assert h.sum() == gray.size
    p = h / gray.size                  # 4. normalisasi, jumlahnya = 1
    cdf = np.cumsum(p)                 # 5. distribusi kumulatif
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(np.arange(256), p, width=1.0, color='gray')
    ax.set_xlabel('Intensitas'); ax.set_ylabel('p(j)')
    ax2 = ax.twinx(); ax2.plot(cdf, 'r-'); ax2.set_ylim(0, 1.02); ax2.set_ylabel('CDF')
    ax.set_title(NIM + ' - Histogram grayscale + CDF ' + judul)
    plt.show()
    print('jumlah p =', round(p.sum(), 6), '| CDF akhir =', round(cdf[-1], 6))

**Tahap 1 – citra contoh `lena.jpg`** (cocokkan dengan gambar histogram RGB pada modul)

In [ ]:
lena = muat(CONTOH + '/lena.jpg')
plot_hist_rgb(lena, '- lena.jpg (tahap 1)')
plot_hist_gray(muat(CONTOH + '/lena.jpg', gray=True), '- lena.jpg (tahap 1)')

**Tahap 2 – `KTM1b.jpg`** (foto dengan lampu ruangan)

In [ ]:
ktm1b = muat(BASE + '/KTM1b.jpg')
plot_hist_rgb(ktm1b, '- KTM1b.jpg (tahap 2)')
plot_hist_gray(muat(BASE + '/KTM1b.jpg', gray=True), '- KTM1b.jpg (tahap 2)')

## Pertanyaan Praktikum E1

### E1 no. 1 – `histogram_manual` vs `np.histogram` vs `cv.calcHist`
Ketiganya dibandingkan lewat selisih maksimum antar-hasil. Kalau semuanya benar, selisihnya harus tepat nol.

In [ ]:
def banding_tiga(gray, nama):
    t0 = time.perf_counter(); h_man = histogram_manual(gray);                       t_man = time.perf_counter() - t0
    t0 = time.perf_counter(); h_np  = np.histogram(gray, bins=256, range=(0, 256))[0]; t_np = time.perf_counter() - t0
    t0 = time.perf_counter(); h_cv  = cv.calcHist([gray], [0], None, [256], [0, 256]).ravel().astype(np.int64); t_cv = time.perf_counter() - t0
    d1 = np.abs(h_man - h_np).max()
    d2 = np.abs(h_man - h_cv).max()
    d3 = np.abs(h_np  - h_cv).max()
    print('=== %s (%d x %d piksel) ===' % (nama, gray.shape[1], gray.shape[0]))
    print('selisih maks manual  vs np.histogram :', d1)
    print('selisih maks manual  vs cv.calcHist  :', d2)
    print('selisih maks np.hist vs cv.calcHist  :', d3)
    print('waktu (detik)  manual: %.3f | np.histogram: %.5f | cv.calcHist: %.5f' % (t_man, t_np, t_cv))
    assert max(d1, d2, d3) == 0, 'Selisih tidak nol, periksa kode!'
    print('-> ketiganya identik (selisih = 0)\n')

banding_tiga(muat(CONTOH + '/lena.jpg', gray=True), 'lena.jpg (tahap 1)')
banding_tiga(muat(BASE + '/KTM1b.jpg', gray=True), 'KTM1b.jpg (tahap 2)')

### E1 no. 2 – Histogram ternormalisasi p(j) dan kurva CDF, KTM1a sampai KTM1d

In [ ]:
kondisi = {'KTM1a': 'ruangan gelap', 'KTM1b': 'lampu ruangan',
           'KTM1c': 'ruangan + flash', 'KTM1d': 'luar ruangan, matahari'}
data, kurva = {}, {}
for n in BERKAS_KTM:
    p = BASE + '/' + n + '.jpg'
    if os.path.exists(p):
        data[n] = muat(p, gray=True)
        kurva[n] = p_cdf(data[n])
    else:
        print('Dilewati (berkas tidak ada):', n)

fig, axs = plt.subplots(2, 2, figsize=(14, 8))
for ax, n in zip(axs.ravel(), data):
    p, cdf = kurva[n]
    ax.bar(np.arange(256), p, width=1.0, color='gray')
    ax.set_xlabel('Intensitas j'); ax.set_ylabel('p(j)')
    ax2 = ax.twinx(); ax2.plot(cdf, 'r-', lw=2); ax2.set_ylim(0, 1.02); ax2.set_ylabel('CDF')
    ax.set_title(NIM + ' - ' + n + ' (' + kondisi[n] + ')')
plt.tight_layout(); plt.show()

def statistik(g, cdf):
    return {
        'rerata'        : g.mean(),
        'median'        : int(np.searchsorted(cdf, 0.5)),
        'std'           : g.std(),
        'p5'            : int(np.searchsorted(cdf, 0.05)),
        'p95'           : int(np.searchsorted(cdf, 0.95)),
        'CDF(63) %'     : cdf[63] * 100,
        'CDF(127) %'    : cdf[127] * 100,
        '% piksel >=250': max(0.0, (1 - cdf[249]) * 100),
    }
tabel_stat = pd.DataFrame({n: statistik(data[n], kurva[n][1]) for n in data}).T.round(2)
tabel_stat.insert(0, 'kondisi', [kondisi[n] for n in tabel_stat.index])
display(tabel_stat)

gelap  = tabel_stat['rerata'].astype(float).idxmin()
terang = tabel_stat['rerata'].astype(float).idxmax()
print('Citra paling gelap  :', gelap,  '-> rerata %.1f, median %d' % (tabel_stat.loc[gelap, 'rerata'],  tabel_stat.loc[gelap, 'median']))
print('Citra paling terang :', terang, '-> rerata %.1f, median %d' % (tabel_stat.loc[terang, 'rerata'], tabel_stat.loc[terang, 'median']))
print('Pada intensitas 63 (seperempat terendah), CDF %s sudah %.1f%% sedangkan CDF %s baru %.1f%%.' %
      (gelap, kurva[gelap][1][63] * 100, terang, kurva[terang][1][63] * 100))
print('Piksel hampir putih (>=250): %s %.2f%%, %s %.2f%%.' %
      (gelap, tabel_stat.loc[gelap, '% piksel >=250'], terang, tabel_stat.loc[terang, '% piksel >=250']))
urut = list(tabel_stat['rerata'].astype(float).sort_values().index)
print('Urutan kecerahan (gelap -> terang):', ' < '.join(urut))

**Jawaban E1 no. 2**

Kurva CDF adalah jumlah kumulatif dari p(j), jadi selalu naik dari 0 ke 1 dan bentuknya langsung menunjukkan ke mana piksel-piksel citra "menumpuk".

- **Citra paling gelap** (dari tabel di atas): CDF naik sangat cepat di sisi kiri. Sebagian besar piksel sudah terhimpun pada intensitas rendah, sehingga kurva menempel ke bagian atas sejak nilai j yang kecil, lalu mendatar. Sisi kanan histogram nyaris kosong. Ini sesuai kondisi pemotretan di ruangan gelap: cahaya yang sampai ke sensor sedikit, kartu dan latar sama-sama tercatat sebagai nilai rendah, dan rentang intensitas yang terpakai sempit sehingga kontras rendah.
- **Citra paling terang**: CDF justru mendatar dan tetap rendah di sisi kiri, baru naik tajam di intensitas tinggi. Bila ada lonjakan di ujung kanan (dekat 255), itu tanda piksel jenuh (overexposed), yang biasanya muncul pada pantulan flash atau sinar matahari langsung di permukaan kartu yang mengilap. Detail di area jenuh sudah hilang dan tidak bisa dipulihkan.
- Selisih posisi "lutut" kurva (median dan CDF(63) pada tabel) adalah ukuran kuantitatif dari beda pencahayaan tersebut. Untuk citra dengan pencahayaan lampu ruangan, kurva biasanya berada di antara keduanya dan lebih landai, karena intensitas tersebar lebih luas.

Catatan: urutan kecerahan tiap KTM bergantung pada foto Anda. Kalau KTM1c (flash) ternyata lebih terang dari KTM1d (matahari) atau sebaliknya, gunakan hasil tabel, jangan asumsi kondisi. Sesuaikan kalimat di atas dengan hasil `gelap` dan `terang` yang tercetak.

### E1 no. 3 – Jumlah bin `PARAM['bins']` vs 256 bin (KTM1b)

In [ ]:
g1b = muat(BASE + '/KTM1b.jpg', gray=True)
nb = PARAM['bins']

fig, axs = plt.subplots(1, 2, figsize=(15, 4))
for ax, b in zip(axs, [nb, 256]):
    cnt, edges = np.histogram(g1b, bins=b, range=(0, 256))
    ax.bar(edges[:-1], cnt, width=np.diff(edges), align='edge', color='gray')
    ax.set_xlabel('Intensitas'); ax.set_ylabel('Jumlah piksel')
    ax.set_title(NIM + ' - KTM1b, %d bin' % b)
plt.tight_layout(); plt.show()

c256 = np.bincount(g1b.ravel(), minlength=256)
lebar = 256 // nb
grup = c256.reshape(nb, lebar).sum(axis=1)
puncak = int(c256.argmax()); bin_puncak = puncak // lebar
print('Setiap bin pada %d bin menggabungkan %d level intensitas.' % (nb, lebar))
print('Level kosong (tak ada piksel) pada 256 bin : %d dari 256' % (c256 == 0).sum())
print('Puncak tertinggi pada 256 bin: level %d (%.2f%% piksel)' % (puncak, c256[puncak] / g1b.size * 100))
print('Pada %d bin puncak itu hanya terlihat sebagai bin level %d-%d (%.2f%% piksel)' %
      (nb, bin_puncak * lebar, bin_puncak * lebar + lebar - 1, grup[bin_puncak] / g1b.size * 100))
print('Piksel di level 255 : %.2f%% | di bin terakhir (%d-255): %.2f%%' % (c256[255] / g1b.size * 100, 256 - lebar, grup[-1] / g1b.size * 100))
print('Piksel di level 0   : %.2f%% | di bin pertama (0-%d)  : %.2f%%' % (c256[0] / g1b.size * 100, lebar - 1, grup[0] / g1b.size * 100))
print('Variasi antar-level bertetangga (256 bin): rerata |selisih| = %.1f%% dari rerata jumlah' %
      (np.abs(np.diff(c256)).mean() / c256.mean() * 100))

**Jawaban E1 no. 3**

Dengan 64 bin, tiap batang mewakili empat level intensitas berurutan. Jumlah pikselnya dijumlahkan, jadi yang hilang adalah **resolusi intensitas**:

- **Posisi persis puncak** tidak lagi terlihat. Pada 256 bin kita tahu level tepatnya (mis. level yang tercetak di atas), sedangkan pada 64 bin kita hanya tahu puncak ada di salah satu dari empat level.
- **Struktur halus** seperti lonjakan sempit, celah/level kosong, dan pola bergerigi (biasanya akibat kompresi JPEG atau pemrosesan otomatis kamera HP) tersamarkan dan histogram tampak lebih mulus daripada kenyataannya.
- **Piksel yang jenuh** di 0 atau 255 tercampur dengan level di dekatnya, sehingga kita sulit membedakan piksel yang benar-benar hitam/putih penuh dari yang hanya sangat gelap/sangat terang. Padahal ini informasi penting untuk menilai over/underexposure.
- Sebagai gantinya, bentuk global (letak massa, lebar sebaran) tetap terbaca dan grafik lebih ringkas serta tidak terlalu berisik, jadi 64 bin cukup untuk ringkasan kasar, sedangkan 256 bin diperlukan untuk analisis yang teliti, misalnya sebelum ekualisasi.

---
# E-2 Percobaan Histogram Equalization

### Pemanasan: verifikasi contoh 3-bit pada modul
Sebelum masuk ke citra sungguhan, saya cek dulu contoh hitungan manual di modul (Tabel 1, citra 3-bit 64×64).

Catatan: di modul tertulis "delapan level lama menyusut menjadi **enam** level baru", padahal daftar nilai barunya hanya 1, 3, 5, 6, dan 7, yaitu **lima** level. Hasil kode di bawah menunjukkan lima level (level 0, 2, dan 4 tidak muncul, dan r3 serta r4 sama-sama menjadi 6).

In [ ]:
nk = np.array([790, 1023, 850, 656, 329, 245, 122, 81])
MN = nk.sum()
print('MN =', MN)

p_tepat  = nk / MN
p_bulat  = np.round(p_tepat, 2)              # modul memakai p(rk) yang sudah dibulatkan 2 desimal
s_tepat  = 7 * np.cumsum(p_tepat)
s_modul  = 7 * np.cumsum(p_bulat)
tab = pd.DataFrame({'rk': range(8), 'nk': nk, 'p(rk)': p_bulat,
                    'CDF': np.cumsum(p_bulat), 'sk=7xCDF (modul)': s_modul.round(2),
                    'sk (tanpa pembulatan p)': s_tepat.round(2),
                    'sk dibulatkan': np.round(s_modul).astype(int)})
display(tab)
assert list(np.round(s_modul).astype(int)) == list(np.round(s_tepat).astype(int)) == [1, 3, 5, 6, 6, 7, 7, 7]
level_baru = sorted(int(v) for v in set(np.round(s_modul).astype(int)))
print('Level baru yang muncul:', level_baru, '->', len(level_baru), 'level')

### E2 no. 1 – Histogram equalization manual (rumus modul)
Rumusnya dari modul: `K = round( C · (2^k − 1) / (w · h) )`, dengan C = distribusi kumulatif jumlah piksel. Untuk citra berwarna, tiap kanal B, G, R diproses sendiri-sendiri.

Saya juga menyiapkan versi kedua, `he_manual_cdfmin`, yaitu rumus yang sama tetapi nilai CDF terkecil (bin tak-kosong pertama) dikurangkan dulu. Versi ini dipakai untuk menjelaskan selisih terhadap `cv.equalizeHist` di no. 2.

In [ ]:
def he_manual(ch):
    """Rumus modul: K = round(C * (L-1) / (w*h))"""
    C = np.cumsum(np.bincount(ch.ravel(), minlength=256))
    lut = np.round(C * 255.0 / ch.size).astype(np.uint8)
    return lut[ch]

def he_manual_cdfmin(ch):
    """Varian yang memakai CDF minimum, sama dengan yang dikerjakan cv.equalizeHist."""
    h = np.bincount(ch.ravel(), minlength=256)
    C = np.cumsum(h); N = ch.size
    pertama = int(np.nonzero(h)[0][0])
    if h[pertama] == N:
        return ch.copy()
    cmin = h[pertama]
    skala = np.float32(255) / np.float32(N - cmin)
    val = np.rint((C - cmin).astype(np.float32) * skala)
    val[np.arange(256) <= pertama] = 0
    return np.clip(val, 0, 255).astype(np.uint8)[ch]

def he_warna(bgr, fungsi=he_manual):
    return cv.merge([fungsi(c) for c in cv.split(bgr)])

def tampil_he(sebelum, sesudah, nama, ktm=False):
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    tampil(axs[0], sebelum, nama + ' sebelum', ktm)
    tampil(axs[1], sesudah, nama + ' sesudah HE', ktm)
    plt.show()
    fig, axs = plt.subplots(2, 3, figsize=(20, 7), sharex=True)
    fig.suptitle(NIM + ' - Histogram RGB ' + nama + ' (atas: sebelum, bawah: sesudah)')
    for r, im in enumerate([sebelum, sesudah]):
        for c, (idx, warna) in enumerate([(2, 'red'), (1, 'green'), (0, 'blue')]):
            axs[r, c].bar(np.arange(256), np.bincount(im[:, :, idx].ravel(), minlength=256), color=warna)
    fig.text(0.09, 0.5, 'Jumlah Kemunculan', va='center', rotation='vertical')
    fig.text(0.5, 0.04, 'Intensitas Warna', ha='center')
    plt.show()

**Tahap 1 – `lena_lc.jpg`** (bandingkan dengan gambar contoh: citra pucat berubah kontrasnya)

In [ ]:
lena_lc = muat(CONTOH + '/lena_lc.jpg')
tampil_he(lena_lc, he_warna(lena_lc), 'lena_lc.jpg (tahap 1)')

**Tahap 2 – `KTM1a.jpg`** (foto di ruangan gelap)

In [ ]:
ktm1a = muat(BASE + '/KTM1a.jpg')
tampil_he(ktm1a, he_warna(ktm1a), 'KTM1a.jpg (tahap 2)', ktm=True)

### E2 no. 2 – `cv.equalizeHist` dan selisihnya dengan implementasi manual
Kalau dua implementasi dianggap "sama-sama histogram equalization", selisihnya seharusnya kecil. Di bawah dihitung selisih maksimum kanal demi kanal, untuk versi rumus modul (`he_manual`) maupun versi CDF-minimum (`he_manual_cdfmin`).

In [ ]:
def bandingkan_he(bgr, nama):
    print('=== %s ===' % nama)
    baris = []
    for i, nm in enumerate('BGR'):
        ch = bgr[:, :, i]
        ref = cv.equalizeHist(ch).astype(int)
        d_modul  = np.abs(he_manual(ch).astype(int) - ref)
        d_cdfmin = np.abs(he_manual_cdfmin(ch).astype(int) - ref)
        baris.append({'kanal': nm,
                      'selisih maks (rumus modul)': d_modul.max(),
                      '% piksel beda (rumus modul)': round((d_modul > 0).mean() * 100, 2),
                      'selisih maks (CDF-min)': d_cdfmin.max()})
    display(pd.DataFrame(baris).set_index('kanal'))

bandingkan_he(lena_lc, 'lena_lc.jpg (tahap 1)')
bandingkan_he(ktm1a,   'KTM1a.jpg (tahap 2)')

**Penyebab selisih (E2 no. 2)**

Selisih pada versi rumus modul tidak nol karena `cv.equalizeHist` tidak memakai rumus `round(C·255/(w·h))` apa adanya. OpenCV mengurangkan dulu CDF dari bin tak-kosong pertama (`cdf_min`) dan membagi dengan `(w·h − cdf_min)`, sehingga intensitas terendah yang benar-benar ada di citra dipetakan tepat ke 0 dan yang tertinggi ke 255. Rumus modul tidak melakukan normalisasi ini, jadi piksel tergelap hanya turun ke `round(255·cdf_min/(w·h))` (belum tentu 0), dan seluruh tabel pemetaan ikut bergeser sedikit.

Bukti bahwa itulah penyebabnya: setelah `cdf_min` disertakan (`he_manual_cdfmin`), selisih maksimum menjadi 0. Besarnya selisih bergantung pada nilai `cdf_min` tiap kanal: makin banyak piksel yang menumpuk di intensitas terendah (seperti pada foto gelap), makin besar pula pergeseran tabel pemetaannya. Kalau selisih di KTM1a lebih besar daripada di `lena_lc.jpg`, itulah alasannya.

### E2 no. 3 dan 4 – Ekualisasi citra berwarna
Empat cara dibandingkan (ekualisasi B, G, R terpisah; kanal Y pada YCrCb; kanal V pada HSV; kanal L pada LAB), ditambah satu baris CLAHE pada kanal Y untuk soal no. 4 (`clipLimit = PARAM['clip']`, `tileGridSize = (PARAM['tile'], PARAM['tile'])`). Pergeseran warna diukur lewat kanal Hue, dihitung melingkar (0 sampai 179 pada OpenCV) lalu dikali 2 agar jadi derajat.

In [ ]:
def geser_hue(asli, hasil):
    h1 = cv.cvtColor(asli,  cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d  = np.abs(h1 - h2)
    d  = np.minimum(d, 180 - d)          # Hue melingkar 0-179
    return d.mean()

def semua_cara(bgr):
    # cara 1: tiap kanal diekualisasi sendiri-sendiri
    he_rgb = cv.merge([cv.equalizeHist(c) for c in cv.split(bgr)])
    # cara 2: hanya kanal terang
    y, cr, cb = cv.split(cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb))
    he_y = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)
    h, sat, v = cv.split(cv.cvtColor(bgr, cv.COLOR_BGR2HSV))
    he_v = cv.cvtColor(cv.merge([h, sat, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)
    l, a, b = cv.split(cv.cvtColor(bgr, cv.COLOR_BGR2LAB))
    he_l = cv.cvtColor(cv.merge([cv.equalizeHist(l), a, b]), cv.COLOR_LAB2BGR)
    # no. 4: CLAHE pada kanal Y
    clahe = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
    cl_y = cv.cvtColor(cv.merge([clahe.apply(y), cr, cb]), cv.COLOR_YCrCb2BGR)
    judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L', 'CLAHE kanal Y']
    return judul, [bgr, he_rgb, he_y, he_v, he_l, cl_y]

def kontras_lokal(gray, k=15):
    x = gray.astype(np.float32)
    m = cv.boxFilter(x, -1, (k, k)); m2 = cv.boxFilter(x * x, -1, (k, k))
    return np.sqrt(np.maximum(m2 - m * m, 0)).mean()

def catatan_otomatis(t, geser, terang0, terang):
    if t == 'Asli':
        return 'citra acuan'
    arah = 'lebih terang' if terang > terang0 + 2 else ('lebih gelap' if terang < terang0 - 2 else 'kecerahan relatif sama')
    w = 'warna nyaris tidak bergeser' if geser < 2 else ('warna bergeser sedikit' if geser < 6 else 'warna bergeser jelas')
    return arah + ', ' + w + ' (cek dengan mata)'

def analisis_warna(bgr, nama, ktm=False, roi=None):
    judul, citra = semua_cara(bgr)
    fig, axs = plt.subplots(1, 6, figsize=(24, 4))
    for ax, c, t in zip(axs, citra, judul):
        tampil(ax, c, t, ktm)
    plt.show()
    terang0 = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY).mean()
    baris = []
    for t, c in zip(judul, citra):
        gr = cv.cvtColor(c, cv.COLOR_BGR2GRAY)
        terang = gr.mean()
        geser = np.nan if t == 'Asli' else geser_hue(bgr, c) * 2
        baris.append({'Cara ekualisasi': t,
                      'Rata-rata pergeseran Hue (derajat)': geser,
                      'Rerata kecerahan sesudah': terang,
                      'Kontras teks (ROI)': kontras_lokal(potong(gr, roi)) if roi else np.nan,
                      'Catatan pengamatan visual': catatan_otomatis(t, geser, terang0, terang)})
    tabel = pd.DataFrame(baris).set_index('Cara ekualisasi')
    print('=== ' + nama + ' ===')
    display(tabel.round(2))
    return tabel

print('Tahap 1: lena.jpg')
tab_lena = analisis_warna(lena, 'lena.jpg (tahap 1)')

**Tahap 2 – `KTM1d.jpg`** (di luar ruangan, cahaya matahari)

In [ ]:
ktm1d = muat(BASE + '/KTM1d.jpg')
tab_1d = analisis_warna(ktm1d, 'KTM1d.jpg (tahap 2)', ktm=True, roi=ROI_TEKS)

# ringkasan otomatis untuk membantu menjawab
cara4 = ['Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
hue = tab_1d.loc[cara4, 'Rata-rata pergeseran Hue (derajat)']
print('\n[no. 1] Pergeseran Hue terbesar :', hue.idxmax(), '= %.2f derajat' % hue.max())
print('[no. 2] Pergeseran Hue kanal terang (Y, V, L):',
      ', '.join('%s %.2f' % (k, hue[k]) for k in ['Kanal Y', 'Kanal V', 'Kanal L']), '(derajat) -> tidak ada yang persis 0')
yvl = tab_1d.loc[['Kanal Y', 'Kanal V', 'Kanal L']]
terpilih = yvl['Rata-rata pergeseran Hue (derajat)'].idxmin()
print('[no. 3] Pergeseran Hue paling kecil di antara Y/V/L:', terpilih)
print('        Kontras teks (ROI) Y/V/L:', ', '.join('%s %.2f' % (k, yvl.loc[k, 'Kontras teks (ROI)']) for k in yvl.index))
cl, ky = tab_1d.loc['CLAHE kanal Y'], tab_1d.loc['Kanal Y']
print('[no. 4] CLAHE kanal Y : Hue %.2f derajat, kecerahan %.1f, kontras teks %.2f' % (cl.iloc[0], cl.iloc[1], cl.iloc[2]))
print('        HE biasa Y    : Hue %.2f derajat, kecerahan %.1f, kontras teks %.2f' % (ky.iloc[0], ky.iloc[1], ky.iloc[2]))

**Jawaban E2 no. 3 dan 4** (rujuk angka pada tabel di atas)

1. **Pergeseran Hue terbesar** biasanya terjadi pada cara *ekualisasi B, G, R terpisah*; nama cara dan nilai persisnya tercetak pada baris `[no. 1]`, jadi tulis sesuai hasil itu. Alasannya, tiap kanal ditarik dengan tabel pemetaannya sendiri. Perbandingan antar-kanal (R:G:B) yang menentukan hue ikut berubah, jadi warna kartu bisa berubah rona (mis. condong ke satu warna). Pada cara Y, V, dan L, informasi warna dipisahkan dari kecerahan sehingga yang diubah hanya komponen terang.
2. **Kenapa kanal terang tetap menggeser Hue** meskipun kanal warna tidak disentuh: setelah kanal terang berubah, citra harus dikembalikan ke BGR. Hasil konversi itu **dibulatkan ke bilangan bulat 8-bit dan dipotong (clip) ke 0 sampai 255**. Kalau kecerahan naik banyak, sebagian warna jenuh jatuh di luar gamut RGB dan terpotong, sehingga rasio B:G:R berubah sedikit. Lalu Hue dihitung lagi dari BGR yang sudah dibulatkan. Pada piksel keabuan atau berwarna pucat (saturasi rendah), selisih 1 tingkat saja pada satu kanal dapat menggeser Hue cukup jauh, sebab Hue hampir tak terdefinisi di sana. Karena itu hasilnya tidak pernah persis nol.
3. **Kanal yang paling cocok untuk KTM**: lihat baris `[no. 3]` (Hue paling stabil) dan kontras teks pada ROI. Kanal L pada LAB dan Y pada YCrCb umumnya paling aman karena kanal kecerahannya dibuat mendekati persepsi mata, sedangkan V pada HSV hanya menarik nilai maksimum RGB tanpa menyentuh saturasi, sehingga warna kartu bisa tampak lebih pekat atau pucat tidak wajar. Perlu diingat, Hue pada cara V nyaris tidak bergeser secara konstruksi (Hue dan saturasi HSV tidak disentuh), jadi angka Hue saja belum cukup untuk memilih; lihat juga kesan visual dan kontras teks. Dasar penilaian yang saya pakai adalah area tulisan (kotak hijau di atas, yaitu NIM/nama) dan area latar kartu: kartu yang bagus adalah yang tulisannya makin jelas tanpa warna latar/logo berubah. Cocokkan pilihan Anda dengan angka `Kontras teks` dan `Hue` di tabel.
4. **CLAHE vs ekualisasi biasa** (baris `[no. 4]`): CLAHE bekerja per blok dengan batas penguatan (`clipLimit`), jadi kenaikan kecerahan dan perubahan warnanya biasanya lebih moderat daripada ekualisasi global. Bandingkan Hue, kecerahan, dan kontras teks pada dua baris yang dicetak, lalu tulis mana yang lebih baik menurut angka Anda.

## Pertanyaan Praktikum E2

### E2 no. 1 – Ekualisasi global pada KTM1a (grayscale)

In [ ]:
def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float('inf') if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)

def entropi(g):
    p, _ = p_cdf(g); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

g1a = muat(BASE + '/KTM1a.jpg', gray=True)
he_m = he_manual(g1a)               # implementasi manual (rumus modul)
he_c = cv.equalizeHist(g1a)         # cv.equalizeHist
print('Selisih maks manual vs cv.equalizeHist pada KTM1a grayscale:', np.abs(he_m.astype(int) - he_c.astype(int)).max(),
      '| dengan varian CDF-min:', np.abs(he_manual_cdfmin(g1a).astype(int) - he_c.astype(int)).max())

# (a)+(b) citra asli, hasil ekualisasi, dan histogramnya dalam satu layout
fig, axs = plt.subplots(2, 3, figsize=(18, 8))
for ax, im, t in zip(axs[0], [g1a, he_m, he_c], ['KTM1a asli (gray)', 'HE manual', 'cv.equalizeHist']):
    tampil(ax, im, t, ktm=True)
for ax, im, t in zip(axs[1], [g1a, he_m, he_c], ['histogram asli', 'histogram HE manual', 'histogram cv.equalizeHist']):
    p, cdf = p_cdf(im)
    ax.bar(np.arange(256), p, width=1.0, color='gray')
    ax2 = ax.twinx(); ax2.plot(cdf, 'r-'); ax2.set_ylim(0, 1.02)
    ax.set_title(NIM + ' - ' + t, fontsize=9)
plt.tight_layout(); plt.show()

# (c) PSNR dan pembanding
blur = cv.GaussianBlur(g1a, (9, 9), 2)   # pembanding: citra dikaburkan, jelas TIDAK lebih terbaca
baris = []
for nama, im in [('Asli', g1a), ('HE (cv.equalizeHist)', he_c), ('Dikaburkan (Gaussian)', blur)]:
    baris.append({'citra': nama,
                  'PSNR terhadap asli (dB)': psnr(g1a, im),
                  'std (kontras global)': im.std(),
                  'entropi (bit)': entropi(im),
                  'kontras lokal': kontras_lokal(im),
                  'kontras lokal teks (ROI)': kontras_lokal(potong(im, ROI_TEKS))})
display(pd.DataFrame(baris).set_index('citra').round(2))
print('cek cv.PSNR:', round(cv.PSNR(g1a, he_c), 2), 'dB (harus sama dengan hitungan manual: %.2f dB)' % psnr(g1a, he_c))

**Jawaban E2 no. 1c**

PSNR mengukur seberapa dekat nilai piksel citra hasil dengan citra asli: makin kecil selisih (MSE), makin besar PSNR. Sedangkan ekualisasi histogram memang **sengaja mengubah nilai intensitas semua piksel** secara besar (menarik yang gelap menjadi jauh lebih terang) supaya kontras naik. Wajar kalau MSE-nya besar dan PSNR-nya kecil, walaupun citranya terlihat lebih jelas. Tabel di atas menunjukkan dua hal sekaligus:

- Citra hasil ekualisasi punya PSNR lebih rendah, tetapi std, entropi, dan kontras lokal (termasuk di area tulisan) lebih tinggi. Artinya informasi lebih mudah dibedakan.
- Citra yang sengaja **dikaburkan** justru mendapat PSNR lebih tinggi daripada citra hasil ekualisasi, padahal tulisannya jelas lebih sulit dibaca.

**Kesimpulan:** PSNR adalah ukuran *kemiripan/fidelitas* terhadap citra referensi, bukan ukuran *keterbacaan*. Ia tidak layak dipakai sendirian untuk menilai apakah tulisan pada KTM lebih terbaca. Ukuran yang lebih sesuai: kontras lokal pada area tulisan, ketajaman tepi, atau yang paling langsung, akurasi OCR/pengamatan manusia. PSNR masih berguna bila tujuannya justru menjaga citra tetap dekat dengan aslinya (mis. kompresi).

### E2 no. 2a – CLAHE pada KTM1a vs ekualisasi global

In [ ]:
clahe = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
g_cl = clahe.apply(g1a)

fig, axs = plt.subplots(2, 3, figsize=(18, 8))
semua = [('KTM1a asli', g1a), ('HE global', he_c), ('CLAHE clip %.1f, tile %d' % (PARAM['clip'], PARAM['tile']), g_cl)]
for ax, (t, im) in zip(axs[0], semua):
    tampil(ax, im, t, ktm=True)
for ax, (t, im) in zip(axs[1], semua):
    tampil(ax, potong(im, ROI_TEKS), 'zoom area tulisan - ' + t)
plt.tight_layout(); plt.show()

baris = []
for t, im in semua:
    r = potong(im, ROI_TEKS)
    baris.append({'citra': t, 'std ROI teks': r.astype(float).std(),
                  'kontras lokal ROI teks': kontras_lokal(r),
                  'varians Laplacian ROI teks': cv.Laplacian(r, cv.CV_64F).var()})
tab2a = pd.DataFrame(baris).set_index('citra')
display(tab2a.round(2))
print('Menurut kontras lokal pada area tulisan, yang paling tinggi:', tab2a['kontras lokal ROI teks'].idxmax())

**Jawaban E2 no. 2a**

Bandingkan dua gambar zoom pada baris bawah dengan mata, lalu cocokkan dengan angka di tabel. Secara konsep, ekualisasi global memakai satu tabel pemetaan untuk seluruh citra. Pada foto yang gelap dengan area terang lokal (misalnya kartu dengan bagian yang tersorot), area gelap tetap kurang tertarik dan yang terang bisa terlalu terang. CLAHE membagi citra menjadi blok (`tile`) dan menguatkan kontras di tiap blok dengan batas `clipLimit`. Hasilnya, kontras di area tulisan (yang biasanya kecil dan lokal) naik lebih terkontrol. Kalau tabel di atas menunjukkan kontras lokal dan varians Laplacian tulisan paling tinggi pada CLAHE dan zoom-nya lebih mudah dibaca, maka NIM dan nama lebih terbaca pada hasil CLAHE. Kalau ternyata sebaliknya pada foto Anda, tulis apa adanya sesuai pengamatan.

### E2 no. 2b – Tiga nilai `clipLimit` lain di sekitar nilai NIM

In [ ]:
c0 = PARAM['clip']
offset = sorted([-1.0, -0.5, 0.5, 1.0, 1.5], key=lambda o: (abs(o), o))
lain = [round(c0 + o, 1) for o in offset if c0 + o >= 1.0][:3]
daftar_clip = sorted([c0] + lain)
print('clipLimit yang dicoba:', daftar_clip, '(nilai NIM =', c0, ')')

def noise_latar(im):
    r = potong(im, ROI_LATAR).astype(np.float32)
    return (r - cv.GaussianBlur(r, (0, 0), 2)).std()

hasil_clip = {}
for c in daftar_clip:
    hasil_clip[c] = cv.createCLAHE(clipLimit=c, tileGridSize=(PARAM['tile'], PARAM['tile'])).apply(g1a)

fig, axs = plt.subplots(3, len(daftar_clip), figsize=(5 * len(daftar_clip), 12))
for j, c in enumerate(daftar_clip):
    t = 'clip %.1f' % c + (' (NIM)' if c == c0 else '')
    tampil(axs[0, j], hasil_clip[c], t, ktm=True)
    tampil(axs[1, j], potong(hasil_clip[c], ROI_TEKS), 'zoom teks, ' + t)
    lat = potong(hasil_clip[c], ROI_LATAR)
    axs[2, j].imshow(lat, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axs[2, j].set_title(NIM + ' - zoom latar (noise), ' + t, fontsize=9); axs[2, j].axis('off')
plt.tight_layout(); plt.show()

baris = []
for c in daftar_clip:
    im = hasil_clip[c]
    kt = kontras_lokal(potong(im, ROI_TEKS)); nz = noise_latar(im)
    baris.append({'clipLimit': c, 'kontras lokal teks': kt, 'noise latar (std high-pass)': nz, 'kontras/noise': kt / max(nz, 1e-6)})
tab2b = pd.DataFrame(baris).set_index('clipLimit')
display(tab2b.round(3))
print('Kandidat terbaik menurut rasio kontras teks terhadap noise latar: clipLimit =', tab2b['kontras/noise'].idxmax())
print('Noise latar terendah -> tertinggi:', ' < '.join(str(c) for c in tab2b['noise latar (std high-pass)'].sort_values().index))

**Jawaban E2 no. 2b**

Semakin besar `clipLimit`, semakin besar penguatan kontras yang diizinkan per blok. Tulisan makin tegas, tetapi noise (butiran sensor, artefak JPEG) ikut diperkuat. Pada foto gelap seperti KTM1a, noise ini paling menonjol di **area latar polos yang gelap** (baris ke-3 pada gambar di atas), karena di area itu tidak ada detail sungguhan dan yang diperkuat hanyalah butiran acak. Tabel memperlihatkan naiknya `noise latar` seiring `clipLimit`.

Nilai terbaik dipilih dengan menimbang dua hal: tulisan sudah cukup terbaca (kontras teks tinggi) dan noise latar belum mengganggu. Angka rasio `kontras/noise` di atas adalah petunjuk awal, tetapi keputusan akhirnya tetap lewat pengamatan visual gambar zoom. Tuliskan nilai yang menurut Anda paling baik beserta alasannya (misalnya: "clip X memberi tulisan paling jelas, sedangkan clip di atasnya mulai menampakkan bintik pada latar kartu").

---
# E-3 Tugas Praktikum Dithering (Floyd–Steinberg)

In [ ]:
def floyd_steinberg(gray, L=2):
    img = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape
    for y in range(H):
        for x in range(W):
            lama = img[y, x]
            baru = np.round(lama / step) * step      # warna palet terdekat
            img[y, x] = baru
            error = lama - baru
            if x + 1 < W:               img[y,   x+1] += error * 7 / 16
            if x > 0 and y + 1 < H:     img[y+1, x-1] += error * 3 / 16
            if y + 1 < H:               img[y+1, x  ] += error * 5 / 16
            if x + 1 < W and y + 1 < H: img[y+1, x+1] += error * 1 / 16
    return np.clip(img, 0, 255).astype(np.uint8)

def fs_warna(bgr, L=2):
    """Untuk citra berwarna: jalankan per kanal B, G, R lalu gabungkan lagi."""
    return cv.merge([floyd_steinberg(c, L) for c in cv.split(bgr)])

### Perhitungan manual satu langkah Floyd–Steinberg dan verifikasinya dengan program

Sesuai tujuan no. 6, saya hitung tangan satu langkah pada piksel tengah patch 3×3 dengan **L = 4** (nilai `PARAM['L']` saya), lalu cocokkan dengan program. Konvensi yang dipakai sama seperti modul: x = kolom, y = baris, dan piksel sebelum (1,1) dianggap sudah diproses.

Patch (nilai sebelum piksel tengah diproses):

| | x=0 | x=1 | x=2 |
|---|---|---|---|
| y=0 | 60 | 110 | 200 |
| y=1 | 90 | **130** | 170 |
| y=2 | 30 | 220 | 250 |

Langkah hitungan untuk piksel (x=1, y=1) yang bernilai 130:

1. `step = 255 / (L − 1) = 255 / 3 = 85`, jadi palet = {0, 85, 170, 255}.
2. `130 / 85 = 1,529` → dibulatkan `2` → warna baru = `2 × 85 = 170`.
3. `error = lama − baru = 130 − 170 = −40`.
4. Sebar error ke tetangga:
   - kanan (x+1, y): `170 + (7/16)(−40) = 170 − 17,5 = 152,5`
   - kiri-bawah (x−1, y+1): `30 + (3/16)(−40) = 30 − 7,5 = 22,5`
   - bawah (x, y+1): `220 + (5/16)(−40) = 220 − 12,5 = 207,5`
   - kanan-bawah (x+1, y+1): `250 + (1/16)(−40) = 250 − 2,5 = 247,5`
5. Pemeriksaan: total yang disebar = −17,5 − 7,5 − 12,5 − 2,5 = **−40** = error (bobot 7+3+5+1 = 16, jadi tidak ada error yang hilang).

In [ ]:
def satu_langkah(patch, y, x, L):
    """Satu langkah Floyd-Steinberg pada piksel (x, y), ditulis langsung dari rumus/pseudo-code modul."""
    im = patch.astype(np.float64).copy()
    H, W = im.shape
    step = 255.0 / (L - 1)
    lama = im[y, x]
    baru = np.round(lama / step) * step
    im[y, x] = baru
    err = lama - baru
    if x + 1 < W:               im[y,   x+1] += err * 7 / 16
    if x > 0 and y + 1 < H:     im[y+1, x-1] += err * 3 / 16
    if y + 1 < H:               im[y+1, x  ] += err * 5 / 16
    if x + 1 < W and y + 1 < H: im[y+1, x+1] += err * 1 / 16
    return im, lama, baru, err

patch = np.array([[60, 110, 200],
                  [90, 130, 170],
                  [30, 220, 250]], dtype=np.float64)
L_ = PARAM['L']
hasil, lama, baru, err = satu_langkah(patch, 1, 1, L_)
print('L =', L_, '| lama =', lama, '| baru =', baru, '| error =', err)
print('Patch setelah satu langkah:'); print(hasil)

if L_ == 4:   # nilai hasil hitung tangan di atas
    tangan = {'baru': 170.0, 'error': -40.0, 'kanan (1,2)': 152.5, 'kiri-bawah (2,0)': 22.5,
              'bawah (2,1)': 207.5, 'kanan-bawah (2,2)': 247.5}
    program = {'baru': baru, 'error': err, 'kanan (1,2)': hasil[1, 2], 'kiri-bawah (2,0)': hasil[2, 0],
               'bawah (2,1)': hasil[2, 1], 'kanan-bawah (2,2)': hasil[2, 2]}
    display(pd.DataFrame({'hitung tangan': tangan, 'program': program}))
    assert all(np.isclose(tangan[k], program[k]) for k in tangan)
    assert np.isclose(hasil[1, 2] - patch[1, 2] + hasil[2, 0] - patch[2, 0] + hasil[2, 1] - patch[2, 1] + hasil[2, 2] - patch[2, 2], err)
    print('Hitungan tangan dan program cocok, total error yang disebar = error piksel.')
else:
    print('PARAM[L] bukan 4, hitung ulang tangan dengan step =', 255 / (L_ - 1))

### E3 no. 1 – Dithering Floyd–Steinberg

**Tahap 1 – `lena.jpg`** (grayscale dengan L = 2 supaya bisa dicocokkan dengan contoh, lalu versi berwarna)

In [ ]:
lena_g = muat(CONTOH + '/lena.jpg', gray=True)
fs_lena = floyd_steinberg(lena_g, L=2)
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
tampil(axs[0], lena_g, 'lena.jpg asli')
tampil(axs[1], fs_lena, 'Floyd-Steinberg, L = 2')
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
tampil(axs[0], lena, 'lena.jpg asli (berwarna)')
tampil(axs[1], fs_warna(lena, PARAM['L']), 'Floyd-Steinberg warna, L = %d' % PARAM['L'])
plt.show()

**Tahap 2 – `KTM1b.jpg`** dengan `L = PARAM['L']`

In [ ]:
ktm1b_g = muat(BASE + '/KTM1b.jpg', gray=True)
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
tampil(axs[0], ktm1b_g, 'KTM1b asli (gray)', ktm=True)
tampil(axs[1], floyd_steinberg(ktm1b_g, PARAM['L']), 'Floyd-Steinberg, L = %d' % PARAM['L'], ktm=True)
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
tampil(axs[0], ktm1b, 'KTM1b asli (warna)', ktm=True)
tampil(axs[1], fs_warna(ktm1b, PARAM['L']), 'Floyd-Steinberg warna, L = %d' % PARAM['L'], ktm=True)
plt.show()

### E3 no. 2 – Grayscale → equalization → dithering, dibanding dithering langsung
Empat citra ditampilkan: asli, dithering langsung, hasil ekualisasi, dan ekualisasi lalu dithering. Tahap 1 memakai `lena_lc.jpg` dengan L = 2 (agar cocok dengan contoh), tahap 2 memakai `KTM1a.jpg` dengan `L = PARAM['L']`.

In [ ]:
def empat_panel(gray, L, nama, ktm=False):
    langsung  = floyd_steinberg(gray, L)
    he        = cv.equalizeHist(gray)
    he_dither = floyd_steinberg(he, L)
    fig, axs = plt.subplots(1, 4, figsize=(20, 5.5))
    for ax, im, t in zip(axs, [gray, langsung, he, he_dither],
                         [nama + ' asli', 'dithering langsung (L=%d)' % L, 'hasil ekualisasi', 'ekualisasi lalu dithering (L=%d)' % L]):
        tampil(ax, im, t, ktm)
    plt.tight_layout(); plt.show()
    return gray, langsung, he, he_dither

print('Tahap 1: lena_lc.jpg, L = 2')
lc_g = muat(CONTOH + '/lena_lc.jpg', gray=True)
_ = empat_panel(lc_g, 2, 'lena_lc.jpg')

In [ ]:
print('Tahap 2: KTM1a.jpg, L =', PARAM['L'])
a_g, a_langsung, a_he, a_he_dither = empat_panel(g1a, PARAM['L'], 'KTM1a', ktm=True)

# zoom area tulisan
fig, axs = plt.subplots(1, 4, figsize=(20, 4))
for ax, im, t in zip(axs, [a_g, a_langsung, a_he, a_he_dither],
                     ['asli', 'dithering langsung', 'hasil ekualisasi', 'ekualisasi lalu dithering']):
    tampil(ax, potong(im, ROI_TEKS), 'zoom teks - ' + t)
plt.tight_layout(); plt.show()

# ukuran keterbacaan: kontras area tulisan setelah dikaburkan sedikit (meniru mata yang merata-ratakan titik dither)
def keterbacaan(im):
    r = potong(im, ROI_TEKS).astype(np.float32)
    halus = cv.GaussianBlur(r, (0, 0), 1.5)
    return halus.std(), (r <= 0).mean() * 100, (r >= 255).mean() * 100

baris = []
for t, im in [('asli', a_g), ('dithering langsung', a_langsung), ('hasil ekualisasi', a_he), ('ekualisasi lalu dithering', a_he_dither)]:
    s, hitam, putih = keterbacaan(im)
    baris.append({'citra': t, 'kontras teks (std setelah blur)': s, '% piksel hitam (0)': hitam, '% piksel putih (255)': putih})
tab_e3 = pd.DataFrame(baris).set_index('citra')
display(tab_e3.round(2))
lebih = 'ekualisasi lalu dithering' if tab_e3.loc['ekualisasi lalu dithering', 'kontras teks (std setelah blur)'] > tab_e3.loc['dithering langsung', 'kontras teks (std setelah blur)'] else 'dithering langsung'
print('Kontras teks lebih tinggi pada urutan:', lebih)
print('Persentase piksel level 0 pada area teks, dithering langsung: %.1f%% vs ekualisasi-lalu-dithering: %.1f%%' %
      (tab_e3.loc['dithering langsung', '% piksel hitam (0)'], tab_e3.loc['ekualisasi lalu dithering', '% piksel hitam (0)']))

**Jawaban E3 no. 2**

Urutan yang lebih baik untuk teks pada KTM gelap adalah **ekualisasi dulu, baru dithering**. Alasannya ada pada cara dithering bekerja: setiap piksel dibulatkan ke level palet terdekat (untuk L = 4, palet 0, 85, 170, 255), dan selisihnya disebar ke tetangga. Pada citra gelap, hampir semua nilai piksel berada di sekitar 0 sampai 60, jadi kebanyakan piksel langsung jatuh ke level 0 atau level terendah. Errornya kecil-kecil sehingga hanya sedikit piksel yang "terdorong" ke level berikutnya. Akibatnya kartu hampir seluruhnya hitam dengan sedikit bintik, dan perbedaan antara huruf dan latar ikut hilang (lihat persentase piksel level 0 di tabel).

Kalau citra diekualisasi dulu, intensitas sudah tersebar ke seluruh rentang 0 sampai 255. Huruf dan latar sekarang berada pada level yang berjauhan, sehingga dithering dapat mewakili keduanya dengan pola titik yang berbeda dan tepi huruf tetap terjaga (error diffusion memang dikenal baik dalam mempertahankan tepi). Karena itu kontras area tulisan setelah dithering lebih tinggi pada urutan ekualisasi → dithering, sesuai angka pada tabel dan gambar zoom di atas.

Kelemahannya, ekualisasi juga menguatkan noise pada latar gelap, sehingga hasil dithering-nya terlihat lebih berbintik daripada versi langsung. Untuk keperluan keterbacaan teks, harga itu biasanya sepadan.

---
## Ringkasan
- Histogram manual, `np.histogram`, dan `cv.calcHist` menghasilkan hitungan yang identik (selisih 0).
- Bentuk CDF pada KTM yang gelap dan yang terang mencerminkan kondisi pencahayaan saat difoto. Mengurangi jumlah bin menghilangkan informasi halus (posisi puncak, level kosong, piksel jenuh).
- Ekualisasi manual berbeda dari `cv.equalizeHist` hanya karena pengurangan `cdf_min` pada versi OpenCV. Setelah disamakan, selisihnya 0.
- Ekualisasi per kanal B, G, R menggeser warna paling besar. Ekualisasi hanya pada kanal terang (Y/V/L) jauh lebih aman, walau Hue tetap sedikit bergeser karena pembulatan dan pemotongan saat kembali ke BGR.
- PSNR tidak cocok sebagai ukuran keterbacaan. CLAHE membantu kontras lokal pada teks, dengan harga noise yang naik bersama `clipLimit`.
- Untuk dithering pada citra gelap, ekualisasi sebelum dithering mempertahankan teks jauh lebih baik daripada dithering langsung.